In [ ]:
import os
import time
import random
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.taup import TauPyModel
from obspy.geodetics import locations2degrees

# ============================
# CONFIG
# ============================
CATALOG = "/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog_CLEAN_MB_ONLY.csv"
OUTDIR  = "/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_copilot"

SUCCESS_LOG = os.path.join(OUTDIR, "success_log.txt")
ERROR_LOG   = os.path.join(OUTDIR, "error_log.txt")
SKIP_LOG    = os.path.join(OUTDIR, "skip_log.txt")

YEAR_START = 2004
YEAR_END   = 2005

MAX_WORKERS = 16
VERBOSE = False

client = Client("EARTHSCOPE")
model = TauPyModel(model="iasp91")

# ============================
# LOAD CATALOG
# ============================
df = pd.read_csv(CATALOG)
df["origin_time"] = df["origin_time"].apply(lambda x: UTCDateTime(x))
df = df[df["origin_time"].apply(lambda t: YEAR_START <= t.year <= YEAR_END)]

print(f"Total event dalam rentang {YEAR_START}-{YEAR_END}: {len(df)}\n")

# ============================
# COUNTERS
# ============================
count_saved = 0
count_nodata = 0
count_skipped = 0
count_error = 0

def status_line():
    return f"Saved: {count_saved} | NoData: {count_nodata} | Skipped: {count_skipped} | Error: {count_error}"

# ============================
# UTILITY FUNCTIONS
# ============================
def already_downloaded(event_time, station, network):
    year = event_time.year
    save_dir = f"{OUTDIR}/{year}/{station}/{network}/"
    if not os.path.exists(save_dir):
        return False
    for f in os.listdir(save_dir):
        if event_time.strftime("%Y%m%dT%H%M%S") in f:
            return True
    return False

def estimate_arrivals(lat, lon, depth, st_lat, st_lon):
    dist_deg = locations2degrees(lat, lon, st_lat, st_lon)
    arrivals = model.get_travel_times(depth, dist_deg, phase_list=["P", "S"])
    p = next((a.time for a in arrivals if a.phase.name == "P"), None)
    s = next((a.time for a in arrivals if a.phase.name == "S"), None)
    return p, s

def is_valid_waveform(st):
    channels = [tr.stats.channel[-1] for tr in st]
    if not all(c in ["N", "E", "Z"] for c in channels):
        return False
    if len(st) != 3:
        return False
    for tr in st:
        if tr.stats.npts < 1000:
            return False
        if max(abs(tr.data)) < 1:
            return False
    return True

# ============================
# WORKER FUNCTION
# ============================
def process_event(row):
    global count_saved, count_nodata, count_skipped, count_error

    origin = row["origin_time"]
    lat, lon, depth = row["Latitude"], row["Longitude"], row["Depth (km)"]

    try:
        inv = client.get_stations(latitude=lat, longitude=lon, maxradius=3.15, level="channel")
    except:
        count_nodata += 1
        return

    if len(inv) == 0:
        count_nodata += 1
        return

    for net in inv:
        for sta in net:

            channels = [ch.code for ch in sta.channels if ch.code[-1] in ["N", "Z", "E"]]
            if len(channels) < 3:
                continue

            if already_downloaded(origin, sta.code, net.code):
                count_skipped += 1
                return

            p, s = estimate_arrivals(lat, lon, depth, sta.latitude, sta.longitude)
            if p is None or s is None:
                count_nodata += 1
                return

            start = origin + p - random.uniform(5, 10)
            end   = origin + s + 5

            try:
                st = client.get_waveforms(net.code, sta.code, "*", "BH?,HH?,EH?,SH?", start, end)
                st = st.select(channel="*N") + st.select(channel="*Z") + st.select(channel="*E")

                if not is_valid_waveform(st):
                    raise Exception("Waveform NZE tidak valid")

                year = origin.year
                save_dir = f"{OUTDIR}/{year}/{sta.code}/{net.code}/"
                os.makedirs(save_dir, exist_ok=True)

                fname = f"{sta.code}_{net.code}_{origin.strftime('%Y%m%dT%H%M%S')}_NZE.mseed"
                st.write(os.path.join(save_dir, fname), format="MSEED")

                count_saved += 1
                return

            except:
                count_error += 1
                return

# ============================
# MULTI-THREAD EXECUTION
# ============================
start_time = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_event, row) for _, row in df.iterrows()]

    for f in tqdm(as_completed(futures), total=len(futures), ncols=100, desc="Downloading NZE"):
        elapsed = time.time() - start_time
        done = count_saved + count_nodata + count_skipped + count_error
        speed = done / max(elapsed, 1e-6)
        eta = (len(df) - done) / max(speed, 1e-6)

        tqdm.write(
            f"Saved: {count_saved} | NoData: {count_nodata} | "
            f"Skipped: {count_skipped} | Error: {count_error} | "
            f"Speed: {speed:.2f} evt/s | ETA: {eta/60:.1f} min"
        )



Total event dalam rentang 2004-2005: 359



Saved: 0 | NoData: 1 | Skipped: 0 | Error: 0 | Speed: 0.64 evt/s | ETA: 9.3 min
Saved: 0 | NoData: 2 | Skipped: 0 | Error: 0 | Speed: 1.28 evt/s | ETA: 4.7 min
Saved: 0 | NoData: 3 | Skipped: 0 | Error: 0 | Speed: 1.73 evt/s | ETA: 3.4 min


Saved: 0 | NoData: 3 | Skipped: 0 | Error: 1 | Speed: 1.80 evt/s | ETA: 3.3 min


Saved: 0 | NoData: 4 | Skipped: 0 | Error: 1 | Speed: 1.95 evt/s | ETA: 3.0 min
Saved: 0 | NoData: 5 | Skipped: 0 | Error: 1 | Speed: 2.29 evt/s | ETA: 2.6 min


Saved: 0 | NoData: 6 | Skipped: 0 | Error: 1 | Speed: 2.47 evt/s | ETA: 2.4 min
Saved: 0 | NoData: 7 | Skipped: 0 | Error: 1 | Speed: 2.76 evt/s | ETA: 2.1 min


...

Saved: 0 | NoData: 66 | Skipped: 0 | Error: 74 | Speed: 6.42 evt/s | ETA: 0.6 min
Saved: 0 | NoData: 66 | Skipped: 0 | Error: 75 | Speed: 6.44 evt/s | ETA: 0.6 min
Saved: 0 | NoData: 67 | Skipped: 0 | Error: 75 | Speed: 6.47 evt/s | ETA: 0.6 min


Saved: 0 | NoData: 68 | Skipped: 0 | Error: 75 | Speed: 6.49 evt/s | ETA: 0.6 min
Saved: 0 | NoData: 68 | Skipped: 0 | Error: 76 | Speed: 6.49 evt/s | ETA: 0.6 min
Saved: 0 | NoData: 68 | Skipped: 0 | Error: 77 | Speed: 6.53 evt/s | ETA: 0.5 min


Saved: 0 | NoData: 68 | Skipped: 0 | Error: 78 | Speed: 6.52 evt/s | ETA: 0.5 min
Saved: 0 | NoData: 68 | Skipped: 0 | Error: 79 | Speed: 6.55 evt/s | ETA: 0.5 min
Saved: 0 | NoData: 68 | Skipped: 0 | Error: 80 | Speed: 6.58 evt/s | ETA: 0.5 min


Saved: 0 | NoData: 69 | Skipped: 0 | Error: 80 | Speed: 6.59 evt/s | ETA: 0.5 min
Saved: 0 | NoData: 69 | Skipped: 0 | Error: 81 | Speed: 6.57 evt/s | ETA: 0.5 min


Saved: 0 | NoData: 69 | Skipped: 0 | Error: 82 | Speed: 6.62 evt/s | ETA: 0.5 min
